# Logistics Pulse - SQL analysis

Answers to the eight business questions, run against the DuckDB database
built by database/load_data.py. Each section states the question, runs
queries and reads the result.

## Key Findings

- **6.77%** of delivered orders arrive after the promised date - 6 534 out of
  96 470.
- The time is lost **in transport**: 629 hours for a late order against 167 for
  an on-time one. No product category stands out and only a handful of sellers
  do, so the problem is mostly about where orders go, not who ships them.
- Distance raises the risk - 4.5% under 100 km to 11.6% above 1 500 km - but it
  does not explain Rio de Janeiro: at the same distance and in the same months,
  Rio ran **26.2% against Sao Paulo's 7.3%**.
- **The problem comes and goes.** Three months - November 2017, February and
  March 2018 - hold **48%** of all late orders.
- Rio is not the worst state by rate, but it carries **22.9% of all late
  orders** and its problem has a clear start and end: normal until October 2017,
  up to 5 times worse than the rest of the country for six months, normal again
  from April 2018. The episode produced **749 late deliveries** beyond what the
  rest of the country's rate would have caused - **11.5% of every late order in
  the data**.
- Delivery then got faster, from about **10 days to 7**, with no upward drift in
  the promised window - so the improvement is real.
- A late delivery in Rio cost **2.39 review points**, and most of the damage is
  done within the first week: 58.6% of orders 4 to 7 days late get one star
  against 6.6% of orders that arrive on time.

In [1]:
import duckdb

con = duckdb.connect("../data/warehouse/logistics_pulse.duckdb", read_only=True)

## 1. Baseline - what is the on-time delivery rate?

Only delivered orders with a known delivery date count. An unknown delay is
not an on-time delivery.

In [2]:
query = """
SELECT
    COUNT(*) AS orders,
    COUNT(*) FILTER (WHERE is_late) AS late_orders,
    ROUND(COUNT(*) FILTER (WHERE is_late) / COUNT(*) * 100, 2) AS late_rate_pct
FROM orders
WHERE is_late IS NOT NULL
AND order_status = 'delivered';
"""

con.execute(query).df()

,orders,late_orders,late_rate_pct
0,96470,6534,6.77


**6.77% of delivered orders arrive late - 6 534 out of 96 470.**

Orders with no delivery date are excluded: their delay is unknown, not zero.
At 93% on time the headline looks healthy, so the rest of this analysis is
about where the delay sits, not how big it is.

## 2. Where - which stage creates the delay?

The longest stage is not the guilty one. What matters is which stage grows
when an order runs late.

In [3]:
query = """
SELECT
    ROUND(MEDIAN(approval_hours) FILTER (WHERE is_late) ,1) AS approval_late,
    ROUND(MEDIAN(approval_hours) FILTER (WHERE NOT is_late),1) AS approval_ontime,
    ROUND(MEDIAN(preparation_hours) FILTER (WHERE is_late),1) AS preparation_late,
    ROUND(MEDIAN(preparation_hours) FILTER (WHERE NOT is_late),1) AS preparation_ontime,
    ROUND(MEDIAN(transit_hours) FILTER (WHERE is_late),1) AS transit_late,
    ROUND(MEDIAN(transit_hours) FILTER (WHERE NOT is_late),1) AS transit_ontime
FROM orders
WHERE is_late IS NOT NULL
AND order_status = 'delivered';
"""

con.execute(query).df()

,approval_late,approval_ontime,preparation_late,preparation_ontime,transit_late,transit_ontime
0,0.4,0.3,73.6,43.0,628.8,166.7


**Transit creates the delay: 629 hours for a late order against 167 for an
on-time one.**

That is 26 days instead of 7. Seller preparation grows from 43 to 74 hours,
payment approval barely moves.

## 3. Why - is the delay concentrated or spread evenly?

Three cuts, closest to transit first: customer region, then sellers, then
product categories.

### 3a. By customer region

Both numbers matter: the rate says how bad a state is, the share of all late
orders says how much it weighs.

In [4]:
query = """
SELECT
    c.customer_state                  AS state,
    COUNT(*)                          AS orders,
    COUNT(*) FILTER (WHERE o.is_late) AS late_orders,
    ROUND(COUNT(*) FILTER (WHERE o.is_late) / COUNT(*) * 100, 2) AS late_rate_pct,
    ROUND(COUNT(*) FILTER (WHERE o.is_late)
          / SUM(COUNT(*) FILTER (WHERE o.is_late)) OVER () * 100, 2) AS share_of_late_pct
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.is_late IS NOT NULL
AND o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY late_orders DESC
"""

con.execute(query).df()

,state,orders,late_orders,late_rate_pct,share_of_late_pct
0,SP,40494,1820,4.49,27.85
1,RJ,12350,1495,12.11,22.88
2,MG,11354,519,4.57,7.94
3,BA,3256,396,12.16,6.06
4,RS,5344,325,6.08,4.97
5,SC,3546,291,8.21,4.45
6,ES,1995,214,10.73,3.28
7,PR,4923,199,4.04,3.05
8,CE,1279,176,13.76,2.69
9,PE,1593,153,9.60,2.34


**Rio de Janeiro has 12.8% of orders but 22.9% of all late ones, at a 12.11%
rate.**

Sao Paulo has more late orders only because 42% of orders go there - its rate
is 4.49%. At the overall 6.77% rate Rio would have 836 late orders instead of
1 495.

### 3b. By seller

One order can hold items from several sellers. A late order is counted for
each of them, so seller totals add up to more than 6 534.

In [5]:
query = """
SELECT
    ot.seller_id,
    COUNT(DISTINCT o.order_id)                  AS orders,
    COUNT(DISTINCT o.order_id) FILTER (WHERE o.is_late)        AS late_orders,
    ROUND(COUNT(DISTINCT o.order_id) FILTER (WHERE o.is_late) / COUNT(DISTINCT o.order_id) * 100, 2) AS late_pct
FROM orders o
JOIN order_items ot ON o.order_id = ot.order_id
WHERE o.is_late IS NOT NULL
AND o.order_status = 'delivered'
GROUP BY ot.seller_id
ORDER BY late_orders DESC
LIMIT 20;
"""

con.execute(query).df()

,seller_id,orders,late_orders,late_pct
0,4a3ca9315b744ce9f8e9374361493884,1772,172,9.71
1,1f50f920176fa81dab994f9023523100,1399,124,8.86
2,4869f7a5dfa277a7dca6462dcf3b52b2,1124,118,10.50
3,ea8482cd71df3c1969d7b9473ff13abc,1132,96,8.48
4,6560211a19b47992c3666cc44a7e94c0,1819,96,5.28
5,7c67e1448b00f6e969d365cea6b010ab,973,89,9.15
6,da8622b14eb17ae2831f4ac5b9dab84a,1311,87,6.64
7,cc419e0650a3c5ba77189a1882b7556a,1651,87,5.27
8,8b321bb669392f5163d04c59e235e066,930,81,8.71
9,955fee9216a65b617aa5c0531780ce60,1261,77,6.11


**Among the largest sellers none stands out: rates run from 5.3% to 10.5%
around a 6.8% baseline.**

This query sorts by number of late orders, so it can only see the biggest
sellers, whose rates sit near the average by arithmetic.

**Next:** the same cut with a volume floor, sorted by rate.

In [6]:
query = """
SELECT
    ot.seller_id,
    COUNT(DISTINCT o.order_id)                            AS orders,
    COUNT(DISTINCT o.order_id) FILTER (WHERE o.is_late)   AS late_orders,
    ROUND(COUNT(DISTINCT o.order_id) FILTER (WHERE o.is_late)
          / COUNT(DISTINCT o.order_id) * 100, 2)          AS late_pct
FROM orders o
JOIN order_items ot ON o.order_id = ot.order_id
WHERE o.is_late IS NOT NULL
  AND o.order_status = 'delivered'
GROUP BY ot.seller_id
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY late_pct DESC
LIMIT 15
"""
con.execute(query).df()

,seller_id,orders,late_orders,late_pct
0,06a2c3af7b3aee5d69171b0e14f0ee87,389,74,19.02
1,88460e8ebdecbfecb5f9601833981930,246,42,17.07
2,1ca7077d890b907f89be8c954a02686a,108,18,16.67
3,e5a3438891c0bfdb9394643f95273d8e,216,34,15.74
4,cd68562d3f44870c08922d380acae552,122,19,15.57
5,2c9e548be18521d1c43cde1c582c6de8,124,19,15.32
6,431af27f296bc6519d890aa5a05fdb11,116,16,13.79
7,f7ba60f8c3f99e7ee4042fdef03b70c4,218,30,13.76
8,8160255418d5aaa7dbdc9f4c64ebda44,380,52,13.68
9,b499c00f28f4b7069ff6550af8c1348a,147,19,12.93


**Six of the 210 sellers with at least 100 orders run above 15%, the worst at
19.0%.**

Together they carry about 120 late deliveries more than the baseline would
give - small next to the 749 from Rio. Sellers matter, but much less than
where the parcel goes.

### 3c. By product category

One order can contain several categories and is counted for each, so the
totals again add up to more than 6 534. Products with no category show as an
empty row.

In [7]:
query = """
SELECT
    p.product_category                                     AS category,
    COUNT(DISTINCT o.order_id)                             AS orders,
    COUNT(DISTINCT o.order_id) FILTER (WHERE o.is_late)    AS late_orders,
    ROUND(COUNT(DISTINCT o.order_id) FILTER (WHERE o.is_late) / COUNT(DISTINCT o.order_id) * 100, 2) AS late_pct
FROM orders o
JOIN order_items ot ON o.order_id = ot.order_id
JOIN products p ON p.product_id = ot.product_id
WHERE o.is_late IS NOT NULL
AND o.order_status = 'delivered'
GROUP BY p.product_category
ORDER BY late_orders DESC
LIMIT 15;
"""
con.execute(query).df()

,category,orders,late_orders,late_pct
0,bed_bath_table,9272,689,7.43
1,health_beauty,8647,649,7.51
2,sports_leisure,7529,495,6.57
3,furniture_decor,6307,449,7.12
4,computers_accessories,6529,417,6.39
5,watches_gifts,5493,406,7.39
6,housewares,5743,308,5.36
7,telephony,4093,291,7.11
8,auto,3809,278,7.30
9,toys,3803,243,6.39


**Category does not explain it either: among the ten with the most late
orders, rates run from 5.4% to 7.5%.**

**Only geography separates. A handful of sellers run well above the baseline
but carry little volume, categories sit flat, and Rio runs at nearly three
times Sao Paulo's rate.**

That matches question 2 - the delay is made in transit, which is a geographic
process. Question 4 tests whether distance explains it.

## 4. Is it really operational - does distance explain the delay?

Distance is measured between zip code centres, so anything under a few tens
of kilometres is noise rather than signal.

### 4a. Late rate by distance band

Orders fall into five bands, measured by the longest distance in the order -
an order is complete only when its farthest item arrives.

In [8]:
query = """
WITH order_distance AS (
    SELECT
        o.order_id,
        o.is_late,
        MAX(ot.distance_km) AS distance_km
    FROM orders o
    JOIN order_items ot ON o.order_id = ot.order_id
    WHERE o.is_late IS NOT NULL
    AND o.order_status = 'delivered'
    GROUP BY o.order_id, o.is_late
    )
SELECT
    CASE
        WHEN distance_km < 100 THEN 'under 100 km'
        WHEN distance_km < 300 THEN '100-300 km'
        WHEN distance_km < 700 THEN '300-700 km'
        WHEN distance_km < 1500 THEN '700-1500 km'
        ELSE 'over 1500 km'
        END AS distance_bucket,
    COUNT(*)                AS orders,
    COUNT(order_id) FILTER (WHERE is_late)        AS late_orders,
    ROUND(COUNT(order_id) FILTER (WHERE is_late) / COUNT(*) * 100, 1) AS late_pct
FROM order_distance
WHERE distance_km IS NOT NULL
GROUP BY distance_bucket
ORDER BY MIN(distance_km);
"""
con.execute(query).df()

,distance_bucket,orders,late_orders,late_pct
0,under 100 km,17795,797,4.5
1,100-300 km,13314,692,5.2
2,300-700 km,36228,2514,6.9
3,700-1500 km,19866,1466,7.4
4,over 1500 km,8790,1020,11.6


**The late rate climbs with distance, from 4.5% under 100 km to 11.6% above
1 500 km.**

It rises in every band, with much the largest step above 1 500 km. It does not
cover Rio, which runs at 12.1% - higher than orders crossing the whole
country.

### 4b. How far does each state actually sit?

The gradient above only helps if Rio's orders travel a normal distance. This
measures it instead of assuming it.

In [9]:
query = """
WITH order_distance AS (
    SELECT
        o.order_id,
        c.customer_state,
        MAX(ot.distance_km) AS distance_km
    FROM orders o
    JOIN order_items ot ON o.order_id = ot.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.is_late IS NOT NULL
      AND o.order_status = 'delivered'
    GROUP BY o.order_id, c.customer_state
)
SELECT
    customer_state,
    COUNT(*)                        AS orders,
    ROUND(MEDIAN(distance_km), 0)   AS median_distance_km
FROM order_distance
WHERE distance_km IS NOT NULL
GROUP BY customer_state
ORDER BY orders DESC
LIMIT 10
"""

con.execute(query).df()

,customer_state,orders,median_distance_km
0,SP,40406,165.0
1,RJ,12300,396.0
2,MG,11317,491.0
3,RS,5327,856.0
4,PR,4899,427.0
5,SC,3537,530.0
6,BA,3238,1425.0
7,ES,1984,762.0
8,GO,1944,805.0
9,DF,1914,867.0


**Distance does not explain Rio. Its median order travels 396 km - a normal
distance - yet it runs at 12.1%.**

States at similar distance do far better: Parana at 427 km runs 4.0%, Minas
Gerais at 491 km runs 4.6%. Bahia fits the gradient at 1 425 km and 12.2%.
Rio does not, and section 6 takes it on its own.

## 5. Is it structural - has the late rate been stable over time?

Two years of data is too short to average by season, so the late rate is read
month by month on a real timeline.

### 5a. Late rate by month

Months are taken from the purchase date, so each order sits in the month it
entered the system.

In [10]:
query = """
SELECT
    date_trunc('month', order_purchase_timestamp)         AS month,
    COUNT(*)                                              AS orders,
    COUNT(*) FILTER (WHERE is_late)                       AS late_orders,
    ROUND(COUNT(*) FILTER (WHERE is_late) / COUNT(*) * 100, 2) AS late_rate_pct
FROM orders
WHERE is_late IS NOT NULL
AND order_status = 'delivered'
GROUP BY month
ORDER BY month
"""
con.execute(query).df()

,month,orders,late_orders,late_rate_pct
0,2016-09-01,1,1,100.00
1,2016-10-01,265,2,0.75
2,2016-12-01,1,0,0.00
3,2017-01-01,750,22,2.93
4,2017-02-01,1653,49,2.96
5,2017-03-01,2546,116,4.56
6,2017-04-01,2303,151,6.56
7,2017-05-01,3545,106,2.99
8,2017-06-01,3135,95,3.03
9,2017-07-01,3872,108,2.79


**The problem is not constant: three months - November 2017, February and March
2018 - hold 48% of all late orders.**

Inside them the late rate is 15.1%, outside them 4.5%. The 2016 months stay in
the table but not in the reading: 267 orders over three separate months.
November 2017 came with a 63% jump in orders, so demand explains it. February
and March 2018 did not - the volume was ordinary.

### 5b. Promised time against actual time

Both are measured from the purchase date, so the promise and the delivery cover
the same window.

In [11]:
query = """
SELECT
    date_trunc('month', order_purchase_timestamp)                      AS month,
    ROUND(MEDIAN(date_diff('hour', order_purchase_timestamp,
                 order_delivered_customer_date) / 24.0), 1)            AS actual_days,
    ROUND(MEDIAN(date_diff('hour', order_purchase_timestamp,
                 order_estimated_delivery_date) / 24.0), 1)            AS promised_days,
    ROUND(promised_days - actual_days, 1)                              AS gap_days
FROM orders
WHERE is_late IS NOT NULL
  AND order_status = 'delivered'
  AND order_purchase_timestamp >= '2017-01-01'
GROUP BY month
ORDER BY month
"""
con.execute(query).df()

,month,actual_days,promised_days,gap_days
0,2017-01-01,10.8,39.0,28.2
1,2017-02-01,11.0,30.2,19.2
2,2017-03-01,10.1,22.6,12.5
3,2017-04-01,12.8,25.5,12.7
4,2017-05-01,9.8,23.4,13.6
5,2017-06-01,10.8,23.5,12.7
6,2017-07-01,10.2,23.0,12.8
7,2017-08-01,9.8,22.6,12.8
8,2017-09-01,10.3,22.2,11.9
9,2017-10-01,10.3,22.4,12.1


**Deliveries got faster after March 2018: the typical order went from about
10 days to 7, while the promised window stayed in its usual range.**

During the crisis months actual time rose to 14 days against an unchanged
promise, and the promise never trends upward. The lower late rate comes from
faster delivery.

### 5c. Episodes in other states

Four states are shown side by side, month by month, so the shape of each one
can be compared.

In [12]:
query = """
SELECT
    date_trunc('month', o.order_purchase_timestamp)                 AS month,
    ROUND(COUNT(*) FILTER (WHERE c.customer_state = 'RJ' AND o.is_late)
          / COUNT(*) FILTER (WHERE c.customer_state = 'RJ') * 100, 1)  AS rj,
    ROUND(COUNT(*) FILTER (WHERE c.customer_state = 'BA' AND o.is_late)
          / COUNT(*) FILTER (WHERE c.customer_state = 'BA') * 100, 1)  AS ba,
    ROUND(COUNT(*) FILTER (WHERE c.customer_state = 'SP' AND o.is_late)
          / COUNT(*) FILTER (WHERE c.customer_state = 'SP') * 100, 1)  AS sp,
    ROUND(COUNT(*) FILTER (WHERE c.customer_state = 'MG' AND o.is_late)
          / COUNT(*) FILTER (WHERE c.customer_state = 'MG') * 100, 1)  AS mg
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.is_late IS NOT NULL
  AND o.order_purchase_timestamp >= '2017-01-01'
GROUP BY month
ORDER BY month
"""
con.execute(query).df()

,month,rj,ba,sp,mg
0,2017-01-01,3.3,4.2,2.8,0.0
1,2017-02-01,3.5,8.6,3.0,0.8
2,2017-03-01,4.3,8.0,2.3,2.3
3,2017-04-01,6.5,14.3,7.0,2.6
4,2017-05-01,2.8,3.4,4.3,1.7
5,2017-06-01,3.3,7.8,3.3,1.4
6,2017-07-01,3.1,4.9,1.8,0.9
7,2017-08-01,3.0,3.3,2.0,2.4
8,2017-09-01,5.1,16.8,3.1,1.0
9,2017-10-01,6.9,10.1,3.3,0.9


**Rio is the only one of the four with a clear start and end: under 7% until
September 2017, up to 35% from October to March, then under 7% again.**

Bahia is above the 6.8% baseline in 12 of 20 months, so its problem is
standing rather than sudden. Sao Paulo and Minas Gerais stay under 5% in most
months.

## 6. What happened in Rio?

Rio is not the worst state by rate, but it carries more late orders than any
state except Sao Paulo, and its rate has a clear start and end.

### 6a. Rio against the rest of the country, month by month

Every other state is pooled into one line, so two series can be read side by
side instead of twenty-seven.

In [13]:
query = """
SELECT
    date_trunc('month', o.order_purchase_timestamp)                    AS month,
    COUNT(*) FILTER (WHERE c.customer_state = 'RJ')                    AS rj_orders,
    ROUND(COUNT(*) FILTER (WHERE c.customer_state = 'RJ' AND o.is_late)
          / COUNT(*) FILTER (WHERE c.customer_state = 'RJ') * 100, 2)  AS rj_late_pct,
    COUNT(*) FILTER (WHERE c.customer_state <> 'RJ')                   AS rest_orders,
    ROUND(COUNT(*) FILTER (WHERE c.customer_state <> 'RJ' AND o.is_late)
          / COUNT(*) FILTER (WHERE c.customer_state <> 'RJ') * 100, 2) AS rest_late_pct,
    ROUND(rj_late_pct / rest_late_pct, 1)                              AS rj_vs_rest
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.is_late IS NOT NULL
  AND o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
GROUP BY month
ORDER BY month
"""
con.execute(query).df()

,month,rj_orders,rj_late_pct,rest_orders,rest_late_pct,rj_vs_rest
0,2017-01-01,91,3.30,659,2.88,1.1
1,2017-02-01,230,3.48,1423,2.88,1.2
2,2017-03-01,370,4.32,2176,4.60,0.9
3,2017-04-01,325,6.46,1978,6.57,1.0
4,2017-05-01,466,2.79,3079,3.02,0.9
5,2017-06-01,399,3.26,2736,3.00,1.1
6,2017-07-01,547,3.11,3325,2.74,1.1
7,2017-08-01,537,2.98,3656,2.90,1.0
8,2017-09-01,591,5.08,3559,4.27,1.2
9,2017-10-01,654,6.88,3824,3.71,1.9


**Rio broke down between October 2017 and March 2018, then went back to
normal.**

Before that it matched the rest of the country. In the worst months it ran up
to 5 times worse. From April 2018 the two lines sit together again.

### 6b. States compared at the same distance

Only orders of 200-600 km bought during the crisis months, so distance and
period are both held still.

In [14]:
query = """
WITH order_distance AS (
    SELECT order_id, MAX(distance_km) AS distance_km
    FROM order_items
    GROUP BY order_id
)

SELECT
    c.customer_state,
    COUNT(*)                                                     AS orders,
    COUNT(*) FILTER (WHERE o.is_late)                            AS late_orders,
    ROUND(COUNT(*) FILTER (WHERE o.is_late) / COUNT(*) * 100, 2) AS late_rate_pct,
    ROUND(MEDIAN(d.distance_km), 0)                              AS median_distance_km
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_distance d ON o.order_id = d.order_id
WHERE o.order_status = 'delivered'
  AND o.is_late IS NOT NULL
  AND d.distance_km BETWEEN 200 AND 600
  AND o.order_purchase_timestamp >= '2017-10-01'
  AND o.order_purchase_timestamp < '2018-04-01'
GROUP BY c.customer_state
HAVING COUNT(*) >= 200
ORDER BY late_rate_pct DESC
"""
con.execute(query).df()

,customer_state,orders,late_orders,late_rate_pct,median_distance_km
0,RJ,3281,858,26.15,365.0
1,SC,784,96,12.24,469.0
2,MG,2863,245,8.56,466.0
3,RS,219,17,7.76,455.0
4,SP,6437,469,7.29,357.0
5,PR,1248,84,6.73,379.0


**At the same distance and in the same months, Rio ran 26.2% late against 7.3%
in Sao Paulo.**

The six median distances sit between 357 and 469 km, so distance is held
still. Rio is 3.6 times worse than Sao Paulo and twice as bad as Santa
Catarina, the next on the list.

### 6c. The size of the episode

Rio is measured against what the rest of the country's rate would have produced
in the same months, so the national peaks are not counted against Rio.

In [15]:
query = """
WITH monthly AS (
    SELECT
        date_trunc('month', o.order_purchase_timestamp)                AS month,
        COUNT(*) FILTER (WHERE c.customer_state = 'RJ')                AS rj_orders,
        COUNT(*) FILTER (WHERE c.customer_state = 'RJ' AND o.is_late)  AS rj_late,
        COUNT(*) FILTER (WHERE c.customer_state <> 'RJ' AND o.is_late)
            / COUNT(*) FILTER (WHERE c.customer_state <> 'RJ')         AS rest_late_rate
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.is_late IS NOT NULL
      AND o.order_purchase_timestamp >= '2017-10-01'
      AND o.order_purchase_timestamp < '2018-04-01'
    GROUP BY month
)

SELECT
    SUM(rj_orders)                                            AS rj_orders,
    SUM(rj_late)                                              AS rj_late,
    ROUND(SUM(rj_orders * rest_late_rate), 0)                 AS expected_late,
    ROUND(SUM(rj_late) - SUM(rj_orders * rest_late_rate), 0)  AS excess_late,
    ROUND((SUM(rj_late) - SUM(rj_orders * rest_late_rate))
          / (SELECT COUNT(*) FILTER (WHERE is_late)
             FROM orders
             WHERE order_status = 'delivered' AND is_late IS NOT NULL)
          * 100, 1)                                           AS pct_of_all_late
FROM monthly
"""
con.execute(query).df()

,rj_orders,rj_late,expected_late,excess_late,pct_of_all_late
0,5005.0,1195.0,446.0,749.0,11.5


**The episode produced 749 late deliveries beyond what the rest of the
country's rate would have caused - 11.5% of all late orders in the data.**

Rio had 1 195 late orders in those months against 446 expected.

## 7. What does a late delivery cost?

Reviews are the only customer reaction in the data, so the cost is measured in
review scores rather than money.

In [16]:
query = """
SELECT
    CASE
        WHEN o.delay_days <= 0  THEN 'on time or early'
        WHEN o.delay_days <= 3  THEN '1-3 days late'
        WHEN o.delay_days <= 7  THEN '4-7 days late'
        WHEN o.delay_days <= 14 THEN '8-14 days late'
        ELSE '15+ days late'
    END                                                            AS delay_bucket,
    COUNT(*)                                                       AS orders,
    ROUND(AVG(r.review_score), 2)                                  AS avg_score,
    ROUND(COUNT(*) FILTER (WHERE r.review_score = 1)
          / COUNT(*) * 100, 1)                                     AS one_star_pct
FROM orders o
JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.is_late IS NOT NULL
GROUP BY delay_bucket
ORDER BY MIN(o.delay_days)
"""
con.execute(query).df()

,delay_bucket,orders,avg_score,one_star_pct
0,on time or early,89443,4.29,6.6
1,1-3 days late,1852,3.29,25.1
2,4-7 days late,1748,2.10,58.6
3,8-14 days late,1446,1.67,70.6
4,15+ days late,1335,1.72,69.0


**Most of the damage is done within a week: 58.6% of orders 4 to 7 days late
get one star, against 6.6% of orders that arrive on time.**

The average score falls from 4.29 to 1.67 and then stops: orders more than two
weeks late score no worse than those two weeks late.

## 8. What should be done?

The recommendation is built from the size of the Rio episode, what it cost in
review scores, when the signal first appeared, and a test of the rule against
every state.

### 8a. What a late delivery cost in Rio

Rio's own scores from the crisis months are used, not the national average.

In [17]:
query = """
SELECT
    o.is_late,
    COUNT(*)                      AS orders,
    ROUND(AVG(r.review_score), 2) AS avg_score
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN reviews r  ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.is_late IS NOT NULL
  AND c.customer_state = 'RJ'
  AND o.order_purchase_timestamp >= '2017-10-01'
  AND o.order_purchase_timestamp < '2018-04-01'
GROUP BY o.is_late
"""
con.execute(query).df()

,is_late,orders,avg_score
0,True,1165,1.79
1,False,3773,4.18


**A late delivery in Rio scored 1.79 against 4.18 for one that arrived on time,
a loss of 2.39 points.**

### 8b. The excess month by month

The same calculation as the episode total, left split by month so the damage
can be lined up against when the signal appeared.

In [18]:
query = """
WITH monthly AS (
    SELECT
        date_trunc('month', o.order_purchase_timestamp)                AS month,
        COUNT(*) FILTER (WHERE c.customer_state = 'RJ')                AS rj_orders,
        COUNT(*) FILTER (WHERE c.customer_state = 'RJ' AND o.is_late)  AS rj_late,
        COUNT(*) FILTER (WHERE c.customer_state <> 'RJ' AND o.is_late)
            / COUNT(*) FILTER (WHERE c.customer_state <> 'RJ')         AS rest_late_rate
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.is_late IS NOT NULL
      AND o.order_purchase_timestamp >= '2017-10-01'
      AND o.order_purchase_timestamp < '2018-04-01'
    GROUP BY month
)

SELECT
    month,
    rj_orders,
    rj_late,
    ROUND(rj_orders * rest_late_rate, 0)            AS expected_late,
    ROUND(rj_late - rj_orders * rest_late_rate, 0)  AS excess_late
FROM monthly
ORDER BY month
"""
con.execute(query).df()

,month,rj_orders,rj_late,expected_late,excess_late
0,2017-10-01,654,45,24.0,21.0
1,2017-11-01,1012,263,103.0,160.0
2,2017-12-01,746,179,36.0,143.0
3,2018-01-01,850,111,40.0,71.0
4,2018-02-01,879,299,97.0,202.0
5,2018-03-01,864,298,145.0,153.0


**426 of the 749 excess late deliveries - 57% - came after November and
December, the first two months Rio ran at twice the rest of the country.**

### 8c. Does the rule hold up on every state?

The rule tested here - at least 100 orders in the month, and either twice the
rest of the country or above 15% while still half again worse - is run against
every state and month, each measured against the country minus itself.

In [19]:
query = """
WITH state_month AS (
    SELECT
        date_trunc('month', o.order_purchase_timestamp) AS month,
        c.customer_state                                AS state,
        COUNT(*)                                        AS orders,
        COUNT(*) FILTER (WHERE o.is_late)               AS late_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.is_late IS NOT NULL
      AND o.order_purchase_timestamp >= '2017-01-01'
    GROUP BY month, state
),

country_month AS (
    SELECT
        month,
        SUM(orders)      AS orders,
        SUM(late_orders) AS late_orders
    FROM state_month
    GROUP BY month
),

compared AS (
    SELECT
        s.month,
        s.state,
        s.orders,
        s.late_orders / s.orders                          AS state_rate,
        (c.late_orders - s.late_orders)
            / NULLIF(c.orders - s.orders, 0)              AS rest_rate
    FROM state_month s
    JOIN country_month c USING (month)
    WHERE s.orders >= 100
)

SELECT
    month,
    state,
    orders,
    ROUND(state_rate * 100, 1)                  AS state_pct,
    ROUND(rest_rate * 100, 1)                   AS rest_pct,
    ROUND(state_rate / NULLIF(rest_rate, 0), 1) AS ratio,
    CASE
        WHEN state_rate >= 0.15 AND state_rate >= 2 * rest_rate THEN 'both'
        WHEN state_rate >= 0.15                                 THEN 'absolute'
        ELSE                                                         'relative'
    END                                         AS triggered_by
FROM compared
WHERE state_rate >= 2 * rest_rate
   OR (state_rate >= 0.15 AND state_rate >= 1.5 * rest_rate)
ORDER BY month, state
"""
con.execute(query).df()

,month,state,orders,state_pct,rest_pct,ratio,triggered_by
0,2017-03-01,SC,104,18.3,4.0,4.6,both
1,2017-06-01,BA,103,7.8,2.9,2.7,relative
2,2017-08-01,RS,296,7.4,2.6,2.9,relative
3,2017-09-01,BA,167,16.8,3.9,4.3,both
4,2017-10-01,BA,158,10.1,4.0,2.6,relative
5,2017-11-01,CE,103,26.2,12.2,2.1,both
6,2017-11-01,ES,168,20.2,12.2,1.7,absolute
7,2017-11-01,RJ,1012,26.0,10.2,2.5,both
8,2017-12-01,RJ,746,24.0,4.9,4.9,both
9,2018-01-01,RJ,850,13.1,4.7,2.8,relative


**The rule fires 24 times in 20 months, about one state a month.**

Rio appears in five months running from November 2017, which matches section 6.
Espirito Santo and Bahia both run from March 2018 onward - episodes this
analysis never looked at.

**Recommendation: track the late rate by state each month against the rest of
the country. Investigate any state with at least 100 orders that month which is
twice as bad as the rest for two months running, or is above 15% and still half
again worse than the rest.**

The first trigger catches a state pulling away from the rest. The second
catches a month when the whole country degrades and nothing looks twice as bad,
and the "half again" condition keeps it off states that were merely average.

Rio crossed the first trigger in November 2017 and stayed over it until March
2018. The rule would have fired at the end of December. 426 of its 749 excess
late deliveries - 57% - came after that, worth about 1 020 review points.

Run against every state, the rule fires 24 times in 20 months. It also flags
Espirito Santo and Bahia from March 2018 onward, which this analysis did not
examine. The three thresholds are chosen, not derived.

The data holds no carrier, no route and no delivery district, so the rule finds
the next episode sooner but cannot explain this one.

## Limitations

- The 6.77% covers delivered orders only. A further 2 965 orders never arrived
  at all, so "93% on time" does not mean "93% of orders arrived".
- The data holds no carrier, no route and no delivery district, so it shows
  where and when deliveries failed, but not why.
- Distance and region cannot be separated here. States that are far away are
  also served differently, so the distance pattern is not proof that distance
  is the cause. Bahia's high rate mixes both.
- The Rio episode covers Black Friday and Christmas, so part of the rise is
  seasonal demand rather than Rio itself.
- Questions 1 to 4 use the whole period, including 267 orders from 2016. From
  question 5 the data starts in January 2017, where the volume becomes steady.
- The data ends in August 2018.
- Review scores stand in for cost. There is no money in this data, so a late
  delivery can only be priced in review points.

## Next steps

- Carrier, route and delivery district would turn "where and when" into "why".
  None of them are in this dataset.
- Outside sources point to one hypothesis worth testing. Cargo theft in Rio
  state peaked in 2017 at 10 599 incidents, a record since 1991. A federal
  security intervention was decreed on 16 February 2018, with cargo theft among
  its targets, and thefts fell 13.4% across 2018. The period, the state and the
  stage all match the episode in this data, but matching timing is not proof.
- The detection rule flags Espirito Santo and Bahia from March 2018 onward,
  episodes this project did not examine.

Sources: [The Rio Times](https://www.riotimesonline.com/brazil-news/rio-de-janeiro/brief-shows-the-dynamics-of-cargo-theft-in-rio-de-janeiro/),
[Insurance Journal](https://www.insurancejournal.com/news/international/2018/05/31/490699.htm),
[PYMNTS](https://www.pymnts.com/news/ecommerce/2018/cargo-crime-wave-brazil-package-delivery-theft/),
[Wikipedia](https://en.wikipedia.org/wiki/2018_federal_intervention_in_Rio_de_Janeiro)